# Rerun the pilot on a free Kaggle GPU

1. Upload this repository (the whole folder, or its zip) as a Kaggle dataset and add it with **Add Input**.
2. In **Session options**, set **Accelerator = GPU T4 x2** and turn **Internet** on (needs a phone-verified account).
3. Use **Save Version → Save & Run All (Commit)** so the run continues in the background (up to 12 hours).
   Running one model per version is safest: set `MODELS` below to `"llama3.1:8b"` or `"qwen2.5:7b"`.
4. When the version finishes, download `outputs.zip` from the **Output** tab.

To continue an interrupted run, add the earlier version's output as an input and set `RESUME = True`.


In [ ]:
# 1. Settings and copy of the repository to a writable folder
MODELS = "qwen2.5:7b"        # "llama3.1:8b", "qwen2.5:7b" or "llama3.1:8b,qwen2.5:7b"
RESUME = False               # True: continue from an earlier outputs/ folder attached as input

import glob, os, shutil
REPO = "/kaggle/working/repo"
OUT = "/kaggle/working/outputs"
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/run_experiment.py", recursive=True)]
assert found, "Repository not found: Add Input > your dataset"
if not os.path.exists(REPO):
    shutil.copytree(found[0], REPO, ignore=shutil.ignore_patterns("outputs", "reproduced", "__pycache__"))
os.makedirs(OUT, exist_ok=True)
if RESUME:
    for d in glob.glob("/kaggle/input/**/outputs/*-ctx8k", recursive=True):
        dest = os.path.join(OUT, os.path.basename(d))
        if not os.path.exists(dest):
            shutil.copytree(d, dest); print("resuming from", d)
print("repository:", REPO)


In [ ]:
# 2. Install Ollama and the Python packages, start the server (about 2 minutes)
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pip -q install -r /kaggle/working/repo/requirements-run.txt
import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version"); print("Ollama ready"); break
    except Exception:
        time.sleep(1)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 3. Run the experiment (progress lines such as 'done m1 base' appear as blocks finish)
import os
os.environ["PYTHONUNBUFFERED"] = "1"
!python -u /kaggle/working/repo/run_experiment.py --models {MODELS} --out {OUT}


In [ ]:
# 4. Compare with the published run and package the outputs
import glob, os
for d in sorted(glob.glob(OUT + "/*-ctx8k")):
    pub = os.path.join(REPO, "results", "raw", os.path.basename(d))
    !python /kaggle/working/repo/pipeline/compare_runs.py {pub} {d}
!cd /kaggle/working && zip -qr outputs.zip outputs
print("DONE: download outputs.zip from the Output tab")
